# Retrieval-Augmented Generation (RAG) using Amazon Bedrock and Amazon OpenSearch
<img src="images/bedrock and opensearch.png" alt="Bedrock and OpenSearch Architecture" width="1000">

## What is Amazon Bedrock?

Amazon Bedrock is a fully managed Generative AI service from AWS that allows developers to build AI-powered applications without managing the underlying infrastructure.


### Key Features
- Provides access to multiple Foundation Models (FMs) through a single API.
- No need to provision or manage GPU infrastructure.
- Supports models from Amazon, Anthropic, Meta, Mistral AI, Cohere and other providers.
- Enables enterprise-grade security and scalability.
- Supports Retrieval-Augmented Generation (RAG), Agents, Guardrails, Knowledge Bases, Prompt Management and Model Evaluation.
- Integrates seamlessly with other AWS services.

### Why Amazon Bedrock?
- Fully managed service.
- Simple API-based access to multiple LLMs.
- Production-ready and highly scalable.
- Secure integration with enterprise data.
- Native support for Knowledge Bases and Agents.

---

# What is Amazon OpenSearch Serverless?

Amazon OpenSearch Serverless is a fully managed search and vector database service that enables semantic search, vector search and traditional keyword search without managing clusters.

### Key Features
- Fully managed vector database.
- Stores embedding vectors generated from documents.
- Supports semantic similarity search.
- Automatically scales based on workload.
- Integrated directly with Amazon Bedrock Knowledge Bases.
- No cluster management required.

### Why use Amazon OpenSearch in RAG?

In a RAG application, documents are first converted into vector embeddings and stored inside a vector database.

When a user asks a question:

1. Convert the question into an embedding.
2. Perform vector similarity search.
3. Retrieve the most relevant document chunks.
4. Pass the retrieved chunks to the LLM.
5. Generate a grounded response.

OpenSearch acts as the retrieval engine of our RAG application.

---

# What is Retrieval-Augmented Generation (RAG)?

Retrieval-Augmented Generation (RAG) is an architecture that combines a Large Language Model (LLM) with an external knowledge source.

Instead of relying only on the model's training data, RAG retrieves relevant information from enterprise documents before generating a response.

### RAG Workflow

User Question

↓

Convert Question into Embedding

↓

Vector Search (Amazon OpenSearch)

↓

Retrieve Relevant Chunks

↓

Send Context + Question to LLM

↓

Generate Final Response

### Advantages of RAG

- Reduces hallucinations.
- Uses latest enterprise documents.
- No need to retrain the LLM.
- Provides more accurate and grounded answers.
- Easy to update knowledge by adding new documents.

---

# Architecture

<img src="images/rag_arch.png" alt="Bedrock and OpenSearch Architecture" width="1000">

This project uses the following AWS services:

- Amazon Bedrock
    - Foundation Model (Amazon Nova Lite)
    - Titan Text Embeddings V2
    - Knowledge Base

- Amazon OpenSearch Serverless
    - Vector Database

- Amazon S3
    - Stores source documents

- Python (Boto3 SDK)
    - Interacts with Bedrock APIs

---

# Project Implementation Steps

We will build this project completely from scratch.

### Step 1
Create an AWS Free Tier Account.

---

### Step 2
Create an IAM User.

Reason:
- Root users should not be used for day-to-day development.
- IAM users provide secure and controlled access to AWS services.

---

### Step 3
Grant Required Permissions

For this demo, assign:

- AmazonBedrockFullAccess
- AmazonS3FullAccess
- AmazonOpenSearchServiceFullAccess

(Optional)

- IAMFullAccess
- CloudWatchLogsFullAccess

> Note:
> For learning purposes, AdministratorAccess can also be assigned. However, in production environments, follow the principle of least privilege.

---

### Step 4
Login using the IAM User.

Do not use the AWS Root User for application development.

---

### Step 5
Open Amazon Bedrock Console.

Explore:

- Foundation Models
- Knowledge Bases
- Agents
- Prompt Management
- Guardrails
- Model Evaluation

---

### Step 6
Create an Amazon Bedrock Knowledge Base.

During creation:

- Select Amazon OpenSearch Serverless as the vector store.
- Choose Titan Text Embeddings V2.
- Create a new S3 bucket (or use an existing one).
- Upload the documents.
- Synchronize the data source.

---

### Step 7
Verify the Knowledge Base

After synchronization:

- Ensure documents are indexed successfully.
- Verify that embeddings have been generated.
- Test retrieval using the Bedrock console.

---

### Step 8
Build the RAG Application

The implementation consists of four major steps:

1. Connect to Amazon Bedrock.
2. Retrieve relevant document chunks.
3. Pass the retrieved context to Amazon Nova Lite.
4. Generate the final response.

---

# Python Libraries

We will primarily use:

- boto3
- json

---

# APIs Used

Throughout this project we will use two different Bedrock APIs.

## 1. Retrieve API

Purpose:

Retrieve the most relevant document chunks from the Knowledge Base.

This API does NOT generate answers.

---

## 2. Invoke Model API

Purpose:

Send the retrieved context along with the user query to the LLM.

This API generates the final response.

---

### Build a Client

In [9]:
# Send and process a document with Amazon Nova on Amazon Bedrock.

import boto3
from botocore.exceptions import ClientError

# Create a Bedrock Runtime client in the AWS Region you want to use.
client = boto3.client("bedrock-runtime", region_name="us-east-1")

# Set the model ID, e.g. Amazon Nova Lite.
model_id = "amazon.nova-lite-v1:0"

### Connect to LLM

In [10]:
# Start a simple text conversation to test if the model loads
usr_msg = "Hey, Whats the net income of amazon in the 2026?"

conversation = [
    {
        "role": "user",
        "content": [
            {"text": usr_msg}
        ],
    }
]

try:
    # Send the message to the model using a basic inference configuration.
    response = client.converse(
        modelId=model_id,
        messages=conversation,
        inferenceConfig={"maxTokens": 500, "temperature": 0.3},
    )

    # Extract and print the response text.
    response_text = response["output"]["message"]["content"][0]["text"]
    print("Success! Model response:")
    print(response_text)

except (ClientError, Exception) as e:
    print(f"ERROR: Can't invoke '{model_id}'. Reason: {e}")
    exit(1)

Success! Model response:
I'm unable to provide real-time or future financial data, including the net income of Amazon in 2026. Financial information for future years is speculative and subject to change based on various factors such as market conditions, company performance, and global economic trends.

For the most accurate and up-to-date financial information, I recommend checking Amazon's official financial reports, investor relations page, or reliable financial news sources. Additionally, financial analysts and market experts often provide forecasts and projections, but these should be taken with caution as they are based on current data and assumptions that may not hold true in the future.


### Connect to OpenSearch

In [ ]:
client = boto3.client(
    "bedrock-agent-runtime",
    region_name="us-east-1"
)

response = client.retrieve(
    knowledgeBaseId="your-kb-id",
    retrievalQuery={
        "text": "Hey, Whats the net income of amazon in the 2026?"
    }
)

for result in response["retrievalResults"]:
    print(result["content"]["text"])

.   • International segment operating income was $1.4 billion, compared with $1.0 billion in first quarter 2025.   • AWS segment operating income was $14.2 billion, compared with $11.5 billion in first quarter 2025.   • Net income increased to $30.3 billion in the first quarter, or $2.78 per diluted share, compared with $17.1 billion, or $1.59 per diluted share, in first quarter 2025.   • First quarter 2026 net income includes pre-tax gains of $16.8 billion included in non-operating income from our investments in Anthropic.   • Operating cash flow increased 30% to $148.5 billion for the trailing twelve months, compared with $113.9 billion for the trailing twelve months ended March 31, 2025.   • Free cash flow decreased to $1.2 billion for the trailing twelve months, driven primarily by a year-over-year increase of $59.3 billion in purchases of property and equipment, net of proceeds from sales and incentives. This increase primarily reflects investments in artificial intelligence. This

In [ ]:
import boto3
import json

# Clients
kb_client = boto3.client("bedrock-agent-runtime", region_name="us-east-1")
runtime_client = boto3.client("bedrock-runtime", region_name="us-east-1")

# User query
query = "Hey, Whats the net income of amazon in the 2026?"

# Step 1: Retrieve from Knowledge Base
response = kb_client.retrieve(
    knowledgeBaseId="your-kb-id",
    retrievalQuery={
        "text": query
    }
)

# Step 2: Create context
context = "\n\n".join(
    [r["content"]["text"] for r in response["retrievalResults"]]
)

# Step 3: Create prompt
prompt = f"""
You are a helpful assistant.

Context:
{context}

Question:
{query}

Answer based only on the context above.
"""

# Step 4: Invoke Amazon Nova Lite
body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
            ]
        }
    ]
}

response = runtime_client.invoke_model(
    modelId="amazon.nova-lite-v1:0",
    body=json.dumps(body)
)

result = json.loads(response["body"].read())

print(result)

{'output': {'message': {'content': [{'text': 'The net income of Amazon in the first quarter of 2026 was $30.3 billion.'}], 'role': 'assistant'}}, 'stopReason': 'end_turn', 'usage': {'inputTokens': 1714, 'outputTokens': 24, 'totalTokens': 1738, 'cacheReadInputTokenCount': 0, 'cacheWriteInputTokenCount': 0}}


In [6]:
result['output']['message']['content'][0]['text']

'The net income of Amazon in the first quarter of 2026 was $30.3 billion.'